# 43｜从零实现 T5：Span Corruption、相对位置偏置与 Encoder–Decoder

本 Notebook 不调用 transformers、nn.Transformer 或 nn.MultiheadAttention，而是用基础 PyTorch 手写 T5 风格的 RMSNorm、相对位置 bucket、多头自注意力、交叉注意力、EncoderBlock、DecoderBlock 与完整 forward。输入端把连续 span 替换为 sentinel，目标端按 sentinel 顺序还原被遮盖文本。

核心不是追求大模型效果，而是建立可审计合同：缩放必须是 \(1/\sqrt{d_h}\)，decoder 不能读取未来 token，padding 不得改变有效输出，teacher forcing 的右移必须与 loss 对齐，shared embedding 必须真正共享同一个 Parameter。

> 边界：全程 CPU、离线、单线程；合成语料上的受控记忆只证明代码链路可训练，不代表真实 T5 的迁移、生成或语言能力。

## 1. 张量、掩码与复杂度合同

- encoder 输入、decoder 输入均为 [B,T] 的 long；mask 为同形 bool，True 表示有效位置。
- hidden 为 [B,T,D]，拆头后为 [B,H,T,d_h]，要求 D 能被 H 整除。
- encoder self-attention 双向；decoder self-attention 同时使用 causal mask 与 padding mask；cross-attention 的 key/value 来自 encoder。
- 每层注意力时间与显存主项是 O(B·H·T²)，FFN 主项是 O(B·T·D·D_ff)。
- sentinel 是有顺序、不可与普通 token 混用的特殊符号；span 必须非空、互不重叠并按起点排序。

In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import torch
from torch import nn
import torch.nn.functional as F

SEED43 = 4307
random.seed(SEED43)
torch.manual_seed(SEED43)
torch.set_num_threads(1)
DEVICE43 = torch.device("cpu")

PAD43, BOS43, EOS43, UNK43 = 0, 1, 2, 3
SENTINELS43 = [4, 5, 6]
VOCAB43 = [
    "<pad>", "<bos>", "<eos>", "<unk>",
    "<extra_id_0>", "<extra_id_1>", "<extra_id_2>",
    "我", "爱", "机器", "学习", "图", "视觉", "检索", "知识", "文本",
    "生成", "理解", "模型", "数据", "搜索", "语言", "系统",
]
TOKEN_TO_ID43 = {token: index for index, token in enumerate(VOCAB43)}
SPECIAL43 = {PAD43, BOS43, EOS43, UNK43, *SENTINELS43}

assert DEVICE43.type == "cpu"
assert torch.get_num_threads() == 1
assert len(VOCAB43) == len(TOKEN_TO_ID43)
assert SENTINELS43 == list(range(4, 7))
assert SPECIAL43.isdisjoint(set(range(7, len(VOCAB43))))
print({"torch": torch.__version__, "device": str(DEVICE43), "vocab": len(VOCAB43)})

## 2. Span corruption：输入压缩，目标按 sentinel 展开

若原序列为 A B C D E，遮盖 [B,C] 与 [E]，encoder 看到 A S0 D S1；target 是 S0 B C S1 E EOS。与逐 token MLM 不同，一个 sentinel 代表一个连续 span，因此 encoder 序列通常变短。

训练数据必须先切分再做随机 corruption，否则同一原文的不同遮盖版本可能跨 train/test，造成近重复泄漏。这里使用固定 span 是为了让 oracle 完全可复现。

`sentinel_ids` 虽然是可注入参数，但实际使用的前 `span_count` 个值必须数量充足、彼此唯一且来自冻结 `SENTINELS43`。不能拿 EOS 等其他 special 或普通 token 充当 sentinel，否则目标无法无歧义地还原 span。


In [ ]:
def span_corrupt43(token_ids, spans, sentinel_ids=SENTINELS43):
    token_ids = [int(x) for x in token_ids]
    spans = [(int(start), int(end)) for start, end in spans]
    sentinel_ids = [int(x) for x in sentinel_ids]
    if len(spans) > len(sentinel_ids):
        raise ValueError("span 数超过 sentinel 数")
    used_sentinels = sentinel_ids[:len(spans)]
    if len(set(used_sentinels)) != len(used_sentinels):
        raise ValueError("每个实际 span 必须使用唯一 sentinel")
    if any(sentinel not in SENTINELS43 for sentinel in used_sentinels):
        raise ValueError("sentinel 必须来自冻结 sentinel 集合，不能使用其他 special/普通 token")
    previous_end = 0
    for index, (start, end) in enumerate(spans):
        if not (0 <= start < end <= len(token_ids)):
            raise ValueError("span 必须是合法的非空半开区间")
        if index and start < previous_end:
            raise ValueError("span 必须有序且不能重叠")
        previous_end = end
    if any(token in SPECIAL43 for token in token_ids):
        raise ValueError("原始正文不能预先包含特殊 token")

    source, target, cursor = [], [], 0
    for sentinel, (start, end) in zip(sentinel_ids, spans):
        source.extend(token_ids[cursor:start])
        source.append(sentinel)
        target.append(sentinel)
        target.extend(token_ids[start:end])
        cursor = end
    source.extend(token_ids[cursor:])
    target.append(EOS43)
    return source, target


probe_raw43 = [7, 8, 9, 10, 11]
probe_source43, probe_target43 = span_corrupt43(probe_raw43, [(1, 3), (4, 5)])
assert probe_source43 == [7, 4, 10, 5]
assert probe_target43 == [4, 8, 9, 5, 11, EOS43]
assert len(probe_source43) < len(probe_raw43)
assert probe_target43.count(4) == 1 and probe_target43.count(5) == 1

for bad_spans in [[(2, 2)], [(2, 4), (3, 5)], [(-1, 2)]]:
    try:
        span_corrupt43(probe_raw43, bad_spans)
        raise AssertionError("非法 span 未被拒绝")
    except ValueError:
        pass

for bad_sentinels43 in [[4], [4, 4], [4, EOS43], [4, 7]]:
    try:
        span_corrupt43(probe_raw43, [(1, 2), (3, 4)], sentinel_ids=bad_sentinels43)
        raise AssertionError("不足、重复或非冻结 sentinel 未被拒绝")
    except ValueError:
        pass

RAW_RECORDS43 = [
    {"id": "r0", "tokens": [7, 8, 9, 10, 21, 18], "spans": [(2, 4)]},
    {"id": "r1", "tokens": [11, 12, 17, 18, 19, 22], "spans": [(1, 3), (4, 5)]},
    {"id": "r2", "tokens": [15, 13, 20, 22, 19, 18], "spans": [(0, 2)]},
    {"id": "r3", "tokens": [14, 11, 18, 17, 16, 21], "spans": [(2, 4), (5, 6)]},
    {"id": "r4", "tokens": [20, 13, 19, 22, 9, 10], "spans": [(1, 2), (4, 6)]},
    {"id": "r5", "tokens": [21, 18, 16, 15, 17, 22], "spans": [(2, 5)]},
]
EXAMPLES43 = []
for record in RAW_RECORDS43:
    source, target = span_corrupt43(record["tokens"], record["spans"])
    EXAMPLES43.append({**record, "source": source, "target": target})

assert len({record["id"] for record in EXAMPLES43}) == len(EXAMPLES43)
assert all(example["target"][-1] == EOS43 for example in EXAMPLES43)
assert all(set(example["source"]).isdisjoint({BOS43, EOS43}) for example in EXAMPLES43)

## 3. RMSNorm 与 T5 相对位置 bucket

RMSNorm 计算 x / sqrt(mean(x²)+eps) 后乘逐维权重，不减均值。相对位置偏置不保存绝对位置 embedding，而是把 key_position-query_position 映射进有限 bucket：近距离逐格保留，远距离按对数压缩。

双向 encoder 把 bucket 一半分给左侧、一半分给右侧；单向 decoder 只编码当前与过去，未来位置最终还会被 causal mask 设为负无穷。max_distance 只影响远距离压缩，并不扩大可见范围。

In [ ]:
def relative_position_bucket43(relative_position, bidirectional, num_buckets=8, max_distance=16):
    if num_buckets < 4 or max_distance <= num_buckets // 2:
        raise ValueError("bucket 配置过小")
    relative_position = torch.as_tensor(relative_position, dtype=torch.long)
    n = -relative_position
    if bidirectional:
        if num_buckets % 2:
            raise ValueError("双向 bucket 数必须为偶数")
        buckets_per_side = num_buckets // 2
        sign_offset = n.lt(0).long() * buckets_per_side
        n = n.abs()
    else:
        buckets_per_side = num_buckets
        sign_offset = torch.zeros_like(n)
        n = n.clamp_min(0)
    max_exact = buckets_per_side // 2
    is_small = n < max_exact
    safe_n = n.float().clamp_min(float(max_exact))
    logarithmic = max_exact + (
        torch.log(safe_n / max_exact)
        / math.log(max_distance / max_exact)
        * (buckets_per_side - max_exact)
    ).long()
    logarithmic = logarithmic.clamp(max=buckets_per_side - 1)
    return sign_offset + torch.where(is_small, n, logarithmic)


class RMSNorm43(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        if x.shape[-1] != self.weight.numel():
            raise ValueError("RMSNorm 最后一维不匹配")
        rms_inverse = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x.float() * rms_inverse).to(x.dtype) * self.weight


class RelativePositionBias43(nn.Module):
    def __init__(self, heads, num_buckets=8, max_distance=16, bidirectional=True):
        super().__init__()
        self.heads = heads
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, heads)

    def forward(self, query_length, key_length):
        query_position = torch.arange(query_length)[:, None]
        key_position = torch.arange(key_length)[None, :]
        relative = key_position - query_position
        buckets = relative_position_bucket43(
            relative, self.bidirectional, self.num_buckets, self.max_distance
        )
        return self.embedding(buckets).permute(2, 0, 1).unsqueeze(0)


rms_probe43 = torch.tensor([[[3.0, 4.0]]])
rms_layer43 = RMSNorm43(2, eps=0.0)
rms_expected43 = rms_probe43 / math.sqrt((9.0 + 16.0) / 2.0)
assert torch.allclose(rms_layer43(rms_probe43), rms_expected43, atol=1e-7)

positions43 = torch.tensor([0, -1, -2, -16, 1, 2, 16])
assert relative_position_bucket43(positions43, True).tolist() == [0, 1, 2, 3, 5, 6, 7]
causal_buckets43 = relative_position_bucket43(torch.tensor([0, -1, -2, -4, -16, 1]), False)
assert causal_buckets43.tolist() == [0, 1, 2, 4, 7, 0]
assert int(relative_position_bucket43(torch.tensor(-10_000), False)) == 7

## 4. 手写多头注意力：scale、bias 与两类 mask 的先后顺序

Q、K、V 先线性投影并拆成 H 个头，score=QKᵀ/sqrt(d_h)+relative_bias。随后屏蔽无效 key，decoder self-attention 再屏蔽上三角未来位置，最后 softmax。query padding 不能只屏蔽 score：还要在输出端归零，否则投影 bias 会让 padding 重新变成非零。

In [ ]:
class ManualAttention43(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        if dim % heads:
            raise ValueError("dim 必须整除 heads")
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def _split(self, x):
        batch, length, _ = x.shape
        return x.view(batch, length, self.heads, self.head_dim).transpose(1, 2)

    def forward(
        self, query, key_value, key_mask, query_mask,
        causal=False, relative_bias=None,
    ):
        if query.ndim != 3 or key_value.ndim != 3:
            raise ValueError("query/key_value 必须是三维")
        batch, query_length, dim = query.shape
        if key_value.shape[0] != batch or key_value.shape[2] != dim or dim != self.dim:
            raise ValueError("attention 张量形状不匹配")
        key_length = key_value.shape[1]
        if key_mask.shape != (batch, key_length) or key_mask.dtype != torch.bool:
            raise ValueError("key_mask 合同错误")
        if query_mask.shape != (batch, query_length) or query_mask.dtype != torch.bool:
            raise ValueError("query_mask 合同错误")
        if not bool(key_mask.any(dim=1).all()):
            raise ValueError("每个样本至少需要一个有效 key")
        if causal and query_length != key_length:
            raise ValueError("本教学实现的 causal attention 要求 Q/K 等长")

        q = self._split(self.q_proj(query))
        k = self._split(self.k_proj(key_value))
        v = self._split(self.v_proj(key_value))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if relative_bias is not None:
            if relative_bias.shape != (1, self.heads, query_length, key_length):
                raise ValueError("relative_bias 形状错误")
            scores = scores + relative_bias.to(scores.dtype)
        visible = key_mask[:, None, None, :].expand(-1, self.heads, query_length, -1)
        if causal:
            visible = visible & torch.ones(
                query_length, key_length, dtype=torch.bool, device=query.device
            ).tril()[None, None]
        weights = torch.softmax(scores.masked_fill(~visible, -torch.inf), dim=-1)
        weights = weights * query_mask[:, None, :, None]
        context = torch.matmul(weights, v).transpose(1, 2).contiguous().view(batch, query_length, dim)
        output = self.out_proj(context) * query_mask.unsqueeze(-1)
        return output, weights


attention_probe43 = ManualAttention43(4, 2)
with torch.no_grad():
    for layer in [
        attention_probe43.q_proj, attention_probe43.k_proj,
        attention_probe43.v_proj, attention_probe43.out_proj,
    ]:
        layer.weight.copy_(torch.eye(4))
        layer.bias.zero_()
x_probe43 = torch.tensor([[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]])
mask_probe43 = torch.ones(1, 2, dtype=torch.bool)
_, weights_probe43 = attention_probe43(
    x_probe43, x_probe43, mask_probe43, mask_probe43, causal=False
)
expected_head0_scores43 = torch.tensor([[1.0, 0.0], [0.0, 1.0]]) / math.sqrt(2.0)
expected_head0_weights43 = torch.softmax(expected_head0_scores43, dim=-1)
assert torch.allclose(weights_probe43[0, 0], expected_head0_weights43, atol=1e-7)
assert torch.allclose(weights_probe43[0, 1], torch.full((2, 2), 0.5), atol=1e-7)

_, causal_weights43 = attention_probe43(
    x_probe43, x_probe43, mask_probe43, mask_probe43, causal=True
)
assert causal_weights43[0, :, 0, 1].abs().max().item() == 0.0
query_padding43 = torch.tensor([[True, False]])
padded_output43, padded_weights43 = attention_probe43(
    x_probe43, x_probe43, mask_probe43, query_padding43
)
assert padded_output43[0, 1].abs().max().item() == 0.0
assert padded_weights43[0, :, 1].abs().max().item() == 0.0

## 5. EncoderBlock、DecoderBlock 与 tied vocabulary projection

每个子层采用 pre-norm：RMSNorm 后进入 attention/FFN，再与残差相加。decoder 依次执行 causal self-attention、encoder-decoder cross-attention、FFN。这里为了清楚省略 dropout。

T5 的 encoder embedding、decoder embedding 与输出词表矩阵共享参数。forward 中直接使用 F.linear(hidden, shared.weight)，因此不是复制数值，而是同一 Parameter 同时接收输入端与输出端梯度。

In [ ]:
class FeedForward43(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.up = nn.Linear(dim, hidden_dim, bias=False)
        self.down = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class EncoderBlock43(nn.Module):
    def __init__(self, dim, heads, hidden_dim):
        super().__init__()
        self.norm1 = RMSNorm43(dim)
        self.attention = ManualAttention43(dim, heads)
        self.norm2 = RMSNorm43(dim)
        self.ff = FeedForward43(dim, hidden_dim)

    def forward(self, x, mask, relative_bias):
        update, _ = self.attention(
            self.norm1(x), self.norm1(x), mask, mask,
            causal=False, relative_bias=relative_bias,
        )
        x = (x + update) * mask.unsqueeze(-1)
        x = (x + self.ff(self.norm2(x))) * mask.unsqueeze(-1)
        return x


class DecoderBlock43(nn.Module):
    def __init__(self, dim, heads, hidden_dim):
        super().__init__()
        self.norm1 = RMSNorm43(dim)
        self.self_attention = ManualAttention43(dim, heads)
        self.norm2 = RMSNorm43(dim)
        self.cross_attention = ManualAttention43(dim, heads)
        self.norm3 = RMSNorm43(dim)
        self.ff = FeedForward43(dim, hidden_dim)

    def forward(self, x, decoder_mask, memory, memory_mask, relative_bias):
        update, _ = self.self_attention(
            self.norm1(x), self.norm1(x), decoder_mask, decoder_mask,
            causal=True, relative_bias=relative_bias,
        )
        x = (x + update) * decoder_mask.unsqueeze(-1)
        update, _ = self.cross_attention(
            self.norm2(x), memory, memory_mask, decoder_mask, causal=False
        )
        x = (x + update) * decoder_mask.unsqueeze(-1)
        x = (x + self.ff(self.norm3(x))) * decoder_mask.unsqueeze(-1)
        return x


class TinyT543(nn.Module):
    def __init__(self, vocab_size, dim=24, heads=4, hidden_dim=48, layers=1):
        super().__init__()
        self.config = {
            "vocab_size": vocab_size, "dim": dim, "heads": heads,
            "hidden_dim": hidden_dim, "layers": layers,
        }
        self.shared = nn.Embedding(vocab_size, dim, padding_idx=PAD43)
        self.encoder_bias = RelativePositionBias43(heads, bidirectional=True)
        self.decoder_bias = RelativePositionBias43(heads, bidirectional=False)
        self.encoder = nn.ModuleList(
            [EncoderBlock43(dim, heads, hidden_dim) for _ in range(layers)]
        )
        self.decoder = nn.ModuleList(
            [DecoderBlock43(dim, heads, hidden_dim) for _ in range(layers)]
        )
        self.encoder_norm = RMSNorm43(dim)
        self.decoder_norm = RMSNorm43(dim)

    def encode(self, source_ids, source_mask):
        if source_ids.dtype != torch.long or source_ids.shape != source_mask.shape:
            raise ValueError("source 合同错误")
        if source_mask.dtype != torch.bool or not bool(source_mask.any(dim=1).all()):
            raise ValueError("每行 source 至少一个有效 token")
        hidden = self.shared(source_ids) * source_mask.unsqueeze(-1)
        bias = self.encoder_bias(source_ids.shape[1], source_ids.shape[1])
        for block in self.encoder:
            hidden = block(hidden, source_mask, bias)
        return self.encoder_norm(hidden) * source_mask.unsqueeze(-1)

    def forward(self, source_ids, source_mask, decoder_ids, decoder_mask):
        if decoder_ids.dtype != torch.long or decoder_ids.shape != decoder_mask.shape:
            raise ValueError("decoder 合同错误")
        if decoder_mask.dtype != torch.bool or not bool(decoder_mask.any(dim=1).all()):
            raise ValueError("每行 decoder 至少一个有效 token")
        memory = self.encode(source_ids, source_mask)
        hidden = self.shared(decoder_ids) * decoder_mask.unsqueeze(-1)
        bias = self.decoder_bias(decoder_ids.shape[1], decoder_ids.shape[1])
        for block in self.decoder:
            hidden = block(hidden, decoder_mask, memory, source_mask, bias)
        hidden = self.decoder_norm(hidden) * decoder_mask.unsqueeze(-1)
        return F.linear(hidden, self.shared.weight)


model43 = TinyT543(len(VOCAB43)).to(DEVICE43)
source_probe43 = torch.tensor([[7, 4, 10]], dtype=torch.long)
source_mask_probe43 = torch.ones_like(source_probe43, dtype=torch.bool)
decoder_probe43 = torch.tensor([[BOS43, 4, 8]], dtype=torch.long)
decoder_mask_probe43 = torch.ones_like(decoder_probe43, dtype=torch.bool)
logits_probe43 = model43(
    source_probe43, source_mask_probe43, decoder_probe43, decoder_mask_probe43
)
assert logits_probe43.shape == (1, 3, len(VOCAB43))
assert sum(parameter is model43.shared.weight for parameter in model43.parameters()) == 1

decoder_changed43 = decoder_probe43.clone()
decoder_changed43[0, 2] = 15
logits_changed43 = model43(
    source_probe43, source_mask_probe43, decoder_changed43, decoder_mask_probe43
)
assert torch.allclose(logits_probe43[:, :2], logits_changed43[:, :2], atol=1e-6)
assert not torch.allclose(logits_probe43[:, 2], logits_changed43[:, 2])

## 6. Teacher forcing、右移与 token 平均 loss

target 为 [S0, span..., EOS]。decoder 输入必须右移成 [BOS, S0, span...]；第 t 个 logits 预测 target[t]。padding 位置用 ignore_index 排除，分母是有效 target token 数，不是 B×T。

推理时没有真实前缀，只能从 BOS 开始反复前向并取最后一个位置；greedy decode 是确定性的教学基线，真实系统还会使用 beam search、长度惩罚、缓存与约束解码。

批量 greedy 的已完成样本后续写 PAD，不重复写 EOS；返回值同时包含 `tokens`、不含 BOS 的生成长度 `lengths` 和 `finished`。这样下游无需猜测首个 EOS 之后的区域，达到长度上限但未生成 EOS 的样本也能被显式识别。


In [ ]:
def pad_sequences43(sequences, pad_value=PAD43):
    max_length = max(len(sequence) for sequence in sequences)
    result = torch.full((len(sequences), max_length), pad_value, dtype=torch.long)
    mask = torch.zeros_like(result, dtype=torch.bool)
    for row, sequence in enumerate(sequences):
        result[row, :len(sequence)] = torch.tensor(sequence)
        mask[row, :len(sequence)] = True
    return result, mask


def shift_right43(target_ids, target_mask):
    if target_ids.dtype != torch.long or target_ids.ndim != 2:
        raise ValueError("target_ids 必须是二维 long")
    if target_ids.shape != target_mask.shape or target_mask.dtype != torch.bool:
        raise ValueError("target/mask 合同错误")
    if target_ids.shape[1] == 0 or not bool(target_mask[:, 0].all()):
        raise ValueError("每行 target 必须有非空有效前缀")
    seen_padding = (~target_mask).cumsum(dim=1) > 0
    if bool((target_mask & seen_padding).any()):
        raise ValueError("target_mask 必须是连续 True 前缀，只允许右 padding")
    shifted = torch.full_like(target_ids, PAD43)
    shifted[:, 0] = BOS43
    shifted[:, 1:] = target_ids[:, :-1]
    shifted_mask = target_mask.clone()
    shifted = shifted.masked_fill(~shifted_mask, PAD43)
    return shifted, shifted_mask


def sequence_loss43(logits, target_ids, target_mask):
    if logits.shape[:2] != target_ids.shape or target_ids.shape != target_mask.shape:
        raise ValueError("loss 形状合同错误")
    if target_mask.dtype != torch.bool or not bool(target_mask.any()):
        raise ValueError("至少需要一个监督 token")
    labels = target_ids.masked_fill(~target_mask, -100)
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)


def greedy_decode43(model, source_ids, source_mask, max_new_tokens):
    if max_new_tokens <= 0:
        raise ValueError("max_new_tokens 必须为正")
    if source_ids.dtype != torch.long or source_ids.ndim != 2:
        raise ValueError("source_ids 必须是二维 long")
    if source_mask.shape != source_ids.shape or source_mask.dtype != torch.bool:
        raise ValueError("source_mask 合同错误")
    generated = torch.full(
        (source_ids.shape[0], 1), BOS43, dtype=torch.long, device=source_ids.device
    )
    finished = torch.zeros(source_ids.shape[0], dtype=torch.bool, device=source_ids.device)
    lengths = torch.zeros(source_ids.shape[0], dtype=torch.long, device=source_ids.device)
    for step in range(1, max_new_tokens + 1):
        decoder_mask = generated.ne(PAD43)
        proposed = model(source_ids, source_mask, generated, decoder_mask)[:, -1].argmax(-1)
        active = ~finished
        next_token = torch.where(active, proposed, torch.full_like(proposed, PAD43))
        newly_finished = active & next_token.eq(EOS43)
        lengths[newly_finished] = step
        generated = torch.cat([generated, next_token[:, None]], dim=1)
        finished |= newly_finished
        if bool(finished.all()):
            break
    lengths[~finished] = max_new_tokens
    return {"tokens": generated, "lengths": lengths, "finished": finished}


target_oracle43 = torch.tensor([[4, 8, EOS43, PAD43]])
target_mask_oracle43 = torch.tensor([[True, True, True, False]])
shifted_oracle43, shifted_mask_oracle43 = shift_right43(target_oracle43, target_mask_oracle43)
assert shifted_oracle43.tolist() == [[BOS43, 4, 8, PAD43]]
assert shifted_mask_oracle43.equal(target_mask_oracle43)
try:
    shift_right43(
        torch.tensor([[4, 8, EOS43, PAD43]]),
        torch.tensor([[True, False, True, False]]),
    )
    raise AssertionError("带洞 target mask 未被拒绝")
except ValueError:
    pass

class GreedyTerminationStub43:
    def __call__(self, source_ids, source_mask, decoder_ids, decoder_mask):
        batch, length = decoder_ids.shape
        logits = torch.full((batch, length, len(VOCAB43)), -1_000.0)
        if length == 1:
            logits[0, -1, EOS43] = 1_000.0
            logits[1, -1, 7] = 1_000.0
        else:
            logits[:, -1, EOS43] = 1_000.0
        return logits

termination_oracle43 = greedy_decode43(
    GreedyTerminationStub43(), torch.tensor([[7], [8]]),
    torch.ones(2, 1, dtype=torch.bool), max_new_tokens=5,
)
assert termination_oracle43["tokens"].tolist() == [
    [BOS43, EOS43, PAD43], [BOS43, 7, EOS43],
]
assert termination_oracle43["lengths"].tolist() == [1, 2]
assert termination_oracle43["finished"].tolist() == [True, True]

small_logits43 = torch.tensor([[[2.0, 0.0], [0.0, 1.0], [9.0, -9.0]]])
small_target43 = torch.tensor([[0, 1, 0]])
small_mask43 = torch.tensor([[True, True, False]])
actual_loss43 = sequence_loss43(small_logits43, small_target43, small_mask43)
manual_loss43 = (
    -F.log_softmax(small_logits43[0, 0], -1)[0]
    -F.log_softmax(small_logits43[0, 1], -1)[1]
) / 2
assert torch.allclose(actual_loss43, manual_loss43, atol=1e-7)

## 7. 受控训练：只验证 teacher-forcing 链路

六条固定样本全部放入一个小 batch，模型可以记忆。记录初始与最终 token loss，并在留出的记录上只检查 greedy 接口和 token 合法性。这里明确不把训练集 loss 下降称为泛化；真实评估需要独立原文、随机 corruption、序列级 exact match 与去重审计。

In [ ]:
TRAIN_IDS43, VALID_IDS43 = ["r0", "r1", "r2", "r3"], ["r4", "r5"]
train_examples43 = [x for x in EXAMPLES43 if x["id"] in TRAIN_IDS43]
valid_examples43 = [x for x in EXAMPLES43 if x["id"] in VALID_IDS43]
train_source43, train_source_mask43 = pad_sequences43([x["source"] for x in train_examples43])
train_target43, train_target_mask43 = pad_sequences43([x["target"] for x in train_examples43])
train_decoder43, train_decoder_mask43 = shift_right43(train_target43, train_target_mask43)

torch.manual_seed(SEED43)
model43 = TinyT543(len(VOCAB43), dim=24, heads=4, hidden_dim=48, layers=1)
optimizer43 = torch.optim.Adam(model43.parameters(), lr=0.025)
with torch.no_grad():
    initial_loss43 = sequence_loss43(
        model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),
        train_target43, train_target_mask43,
    ).item()

history43 = []
model43.train()
for step in range(30):
    optimizer43.zero_grad(set_to_none=True)
    logits43 = model43(
        train_source43, train_source_mask43, train_decoder43, train_decoder_mask43
    )
    loss43 = sequence_loss43(logits43, train_target43, train_target_mask43)
    loss43.backward()
    torch.nn.utils.clip_grad_norm_(model43.parameters(), 1.0)
    optimizer43.step()
    if step in {0, 9, 19, 29}:
        history43.append((step, float(loss43.detach())))

model43.eval()
with torch.no_grad():
    final_loss43 = sequence_loss43(
        model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),
        train_target43, train_target_mask43,
    ).item()
    valid_source43, valid_source_mask43 = pad_sequences43([x["source"] for x in valid_examples43])
    decoded43 = greedy_decode43(model43, valid_source43, valid_source_mask43, max_new_tokens=10)
    generated43 = decoded43["tokens"]

assert math.isfinite(initial_loss43) and math.isfinite(final_loss43)
assert final_loss43 < initial_loss43 * 0.35
assert generated43.shape[0] == len(VALID_IDS43)
assert generated43[:, 0].eq(BOS43).all()
assert int(generated43.min()) >= 0 and int(generated43.max()) < len(VOCAB43)
assert decoded43["lengths"].shape == (len(VALID_IDS43),)
assert bool(((decoded43["lengths"] >= 1) & (decoded43["lengths"] <= 10)).all())
print({"initial_loss": round(initial_loss43, 4), "final_loss": round(final_loss43, 4), "trace": history43})

## 8. 不变性与失败模式

训练 loss 下降不能替代结构测试。下面分别检查：追加 masked padding 不改变原有效位置；decoder 后缀变化不影响前缀；所有参数梯度有限；非法全空 source fail closed。

常见故障包括 causal mask 方向反了、relative_position 正负号颠倒、把 query padding 只屏蔽在 key 侧、target 未右移、把 embedding 复制成独立 head，以及 span 跨切分泄漏。

In [ ]:
model43.eval()
one_source43 = train_source43[:1, :4]
one_mask43 = train_source_mask43[:1, :4]
one_decoder43 = train_decoder43[:1, :4]
one_decoder_mask43 = train_decoder_mask43[:1, :4]
with torch.no_grad():
    base_logits43 = model43(one_source43, one_mask43, one_decoder43, one_decoder_mask43)
    extended_source43 = torch.cat([one_source43, torch.tensor([[19, 20]])], dim=1)
    extended_mask43 = torch.cat([one_mask43, torch.zeros(1, 2, dtype=torch.bool)], dim=1)
    extended_logits43 = model43(
        extended_source43, extended_mask43, one_decoder43, one_decoder_mask43
    )
assert torch.allclose(base_logits43, extended_logits43, atol=2e-5)

prefix_a43 = one_decoder43.clone()
prefix_b43 = one_decoder43.clone()
prefix_b43[0, -1] = 12 if prefix_a43[0, -1].item() != 12 else 13
with torch.no_grad():
    logits_a43 = model43(one_source43, one_mask43, prefix_a43, one_decoder_mask43)
    logits_b43 = model43(one_source43, one_mask43, prefix_b43, one_decoder_mask43)
assert torch.allclose(logits_a43[:, :-1], logits_b43[:, :-1], atol=2e-5)

model43.zero_grad(set_to_none=True)
gradient_loss43 = sequence_loss43(
    model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),
    train_target43, train_target_mask43,
)
gradient_loss43.backward()
finite_gradients43 = [
    torch.isfinite(parameter.grad).all().item()
    for parameter in model43.parameters() if parameter.grad is not None
]
assert finite_gradients43 and all(finite_gradients43)
assert model43.shared.weight.grad is not None
assert model43.shared.weight.grad.abs().sum().item() > 0

try:
    model43.encode(torch.tensor([[PAD43]]), torch.tensor([[False]]))
    raise AssertionError("全空 source 未被拒绝")
except ValueError:
    pass

## 9. 发布制品：包内哈希不是信任锚

调用方若能同时替换 state、manifest 和包内 hash，自签校验没有安全意义。因此 loader 先用包外、只读 publisher registry 中的已发布指纹验证整个 package，再检查内部一致性。canonical state digest 逐 key 绑定 dtype、shape 与原始 bytes；manifest 完整绑定词表、原始数据、span、split、预处理和训练 recipe。

这里的 MappingProxyType 只模拟部署侧只读配置。真实系统应由签名清单、制品仓库不可变 digest、KMS 公钥或透明日志提供信任锚。

In [ ]:
def canonical_json43(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")


def canonical_state_digest43(state):
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        header = {"key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)}
        digest.update(canonical_json43(header))
        digest.update(tensor.numpy().tobytes(order="C"))
    return digest.hexdigest()


def package_fingerprint43(package):
    digest = hashlib.sha256()
    digest.update(canonical_json43(package["manifest"]))
    digest.update(canonical_state_digest43(package["state"]).encode("ascii"))
    return digest.hexdigest()


manifest43 = {
    "subject": "t5-span-corruption-demo@1",
    "architecture": copy.deepcopy(model43.config),
    "vocab": list(VOCAB43),
    "special_ids": {
        "pad": PAD43, "bos": BOS43, "eos": EOS43, "unk": UNK43,
        "sentinels": list(SENTINELS43),
    },
    "dataset": copy.deepcopy(RAW_RECORDS43),
    "split": {"train": list(TRAIN_IDS43), "validation": list(VALID_IDS43)},
    "preprocess": {
        "tokenizer": "frozen-token-id-v1",
        "span_format": "sorted-half-open",
        "decoder_start": BOS43,
        "target_suffix": EOS43,
        "padding": "right-contiguous-mask",
        "sentinel_policy": "unique-prefix-from-frozen-sentinel-set",
        "greedy_finished_fill": "pad",
        "greedy_length": "generated-token-count-excluding-bos",
    },
    "recipe": {
        "seed": SEED43, "optimizer": "Adam", "learning_rate": 0.025,
        "steps": 30, "gradient_clip": 1.0, "objective": "teacher-forcing-token-ce",
    },
}
state43 = {key: value.detach().cpu().clone() for key, value in model43.state_dict().items()}
package43 = {
    "manifest": manifest43,
    "state": state43,
    "internal": {
        "manifest_digest": hashlib.sha256(canonical_json43(manifest43)).hexdigest(),
        "state_digest": canonical_state_digest43(state43),
    },
}
subject43 = manifest43["subject"]
PUBLISHER_REGISTRY43 = MappingProxyType({subject43: package_fingerprint43(package43)})


def load_published_t543(package, subject):
    if subject not in PUBLISHER_REGISTRY43:
        raise ValueError("未知发布 subject")
    if package_fingerprint43(package) != PUBLISHER_REGISTRY43[subject]:
        raise ValueError("publisher registry 指纹不匹配")
    manifest = package["manifest"]
    if manifest["subject"] != subject:
        raise ValueError("subject 不匹配")
    if hashlib.sha256(canonical_json43(manifest)).hexdigest() != package["internal"]["manifest_digest"]:
        raise ValueError("manifest 内部摘要不匹配")
    if canonical_state_digest43(package["state"]) != package["internal"]["state_digest"]:
        raise ValueError("state 内部摘要不匹配")
    if manifest["vocab"] != VOCAB43 or manifest["special_ids"]["sentinels"] != SENTINELS43:
        raise ValueError("词表或 sentinel 合同不匹配")
    all_ids = {record["id"] for record in manifest["dataset"]}
    train_ids = set(manifest["split"]["train"])
    validation_ids = set(manifest["split"]["validation"])
    if train_ids & validation_ids or train_ids | validation_ids != all_ids:
        raise ValueError("split 必须互斥且覆盖数据")
    for record in manifest["dataset"]:
        span_corrupt43(record["tokens"], record["spans"])
    expected_preprocess43 = {
        "tokenizer": "frozen-token-id-v1",
        "span_format": "sorted-half-open",
        "decoder_start": BOS43,
        "target_suffix": EOS43,
        "padding": "right-contiguous-mask",
        "sentinel_policy": "unique-prefix-from-frozen-sentinel-set",
        "greedy_finished_fill": "pad",
        "greedy_length": "generated-token-count-excluding-bos",
    }
    if manifest["preprocess"] != expected_preprocess43:
        raise ValueError("预处理合同不匹配")
    loaded = TinyT543(**manifest["architecture"])
    loaded.load_state_dict(package["state"], strict=True)
    loaded.eval()
    return loaded


loaded43 = load_published_t543(package43, subject43)
with torch.no_grad():
    assert torch.allclose(
        loaded43(one_source43, one_mask43, one_decoder43, one_decoder_mask43),
        model43(one_source43, one_mask43, one_decoder43, one_decoder_mask43),
        atol=1e-7,
    )
assert canonical_state_digest43(state43) == package43["internal"]["state_digest"]

forged43 = copy.deepcopy(package43)
first_key43 = sorted(forged43["state"])[0]
forged43["state"][first_key43].view(-1)[0] += 1.0
forged43["internal"]["state_digest"] = canonical_state_digest43(forged43["state"])
try:
    load_published_t543(forged43, subject43)
    raise AssertionError("重算包内 state hash 的伪造未被拒绝")
except ValueError as error43:
    assert "registry" in str(error43)

whole_replacement43 = copy.deepcopy(package43)
whole_replacement43["manifest"]["recipe"]["steps"] = 71
replacement_key43 = sorted(whole_replacement43["state"])[-1]
whole_replacement43["state"][replacement_key43].view(-1)[-1] -= 0.25
whole_replacement43["internal"]["manifest_digest"] = hashlib.sha256(
    canonical_json43(whole_replacement43["manifest"])
).hexdigest()
whole_replacement43["internal"]["state_digest"] = canonical_state_digest43(
    whole_replacement43["state"]
)
try:
    load_published_t543(whole_replacement43, subject43)
    raise AssertionError("整体替换并重算内部 hash 未被拒绝")
except ValueError as error43:
    assert "registry" in str(error43)

try:
    PUBLISHER_REGISTRY43[subject43] = "attacker"
    raise AssertionError("只读 registry 被修改")
except TypeError:
    pass

## 10. 从教学实现到生产系统

本实现缺少大规模 tokenizer、动态 span 采样、dropout、混合精度、分布式训练、KV cache、beam search、长文本外推与安全生成策略。生产验收还应覆盖：真实数据去重、污染扫描、长度分桶、吞吐与峰值显存、不同 seed 稳定性、分布外输入、版本回滚和签名轮换。

面试中应区分三层回答：数学上解释 relative bucket 与 causal mask；代码上说明 shape 和 forward；工程上说明 split、可观测性、发布信任锚及受控实验不等于泛化。

原始资料：

- [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer（T5）](https://jmlr.org/papers/v21/20-074.html)
- [T5 官方代码仓库](https://github.com/google-research/text-to-text-transfer-transformer)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)